# Object Lifetime, Memory, and Garbage Collection
Objects do not simply appear and disappear by magic. They are created, referenced, shared, sometimes recycled, and eventually collected. This notebook slows that process down so you can picture what Python is tracking while your code runs.

When people struggle with **Object Lifetime, Memory, and Garbage Collection**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Understand object lifetime in practical terms
- Know what reference counting does
- See why cycles require extra help
- Separate memory cleanup from resource cleanup


An object stays alive as long as something still refers to it. Those references may come from names, containers, object attributes, frames, closures, or global structures. Lifetime is really about reachability.


In [1]:
import sys
value = []
print(sys.getrefcount(value))
other = value
print(sys.getrefcount(value))
del other
print(sys.getrefcount(value))


2
3
2


At the bytecode level you can often see the creation of objects, the storing of references, and the return path. It will not show garbage collection directly, but it does show how names receive references.


In [2]:
import dis

def life():
    data = [1, 2, 3]
    other = data
    return other

dis.dis(life)


  3           0 RESUME                   0

  4           2 BUILD_LIST               0
              4 LOAD_CONST               1 ((1, 2, 3))
              6 LIST_EXTEND              1
              8 STORE_FAST               0 (data)

  5          10 LOAD_FAST                0 (data)
             12 STORE_FAST               1 (other)

  6          14 LOAD_FAST                1 (other)
             16 RETURN_VALUE


When the reference count drops to zero, the object can often be cleaned up immediately.

If two unreachable objects point to each other, their reference counts may never reach zero by themselves.

This is why Python has both ideas in play.

A file descriptor or network connection should be managed explicitly, often with a context manager.


These two objects keep each other alive until the garbage collector notices the cycle is unreachable.


In [3]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None

a = Node("a")
b = Node("b")
a.other = b
b.other = a
print(gc.collect())


34


The inner function still holds onto the captured list even after the outer function finishes.


In [4]:
def factory():
    payload = [1, 2, 3]
    def inner():
        return payload
    return inner

fn = factory()
print(fn())


[1, 2, 3]


The `with` statement is often the simplest answer for resource lifetime.


In [5]:
with open("gc_demo.txt", "w", encoding="utf-8") as f:
    f.write("safe cleanup")
print("done")


done


This is not only a classroom topic. **Object Lifetime, Memory, and Garbage Collection** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Using “garbage collection” as a hand-wavy answer to every lifetime question
- Confusing object lifetime with resource lifetime
- Ignoring cycles when teaching object references


- How does CPython usually manage object memory?
- Why are cyclic references special?
- Why is `with open(...)` still useful if Python has garbage collection?


- Reachability controls lifetime.
- Reference counting is common in CPython.
- Cycles need extra collection.
- Resources should still be managed explicitly.


If this notebook did its job, **Object Lifetime, Memory, and Garbage Collection** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Object Lifetime, Memory, and Garbage Collection is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Object Lifetime, Memory, and Garbage Collection, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x000002358E2B7DC0, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_23844\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST        

To deepen object lifetime understanding, keep asking where the last reference might be hiding. Sometimes it is obvious. Sometimes it is inside a closure cell, a global cache, a container, an object attribute, or a traceback. The object is alive because something can still reach it.

That is why memory questions are often really graph questions. You are trying to understand reachability, not only syntax.


In [7]:
import gc

def make_holder():
    payload = {'data': [1, 2, 3]}
    def inner():
        return payload
    return inner

fn = make_holder()
print('closure cells:', fn.__closure__)
print('closure contents:', fn.__closure__[0].cell_contents)
print('tracked by gc:', gc.is_tracked(fn.__closure__[0].cell_contents))


closure cells: (<cell at 0x000002358E427010: dict object at 0x000002358E47E480>,)
closure contents: {'data': [1, 2, 3]}
tracked by gc: True


To go beyond beginner understanding here, you want to think in reference graphs rather than in isolated statements. Objects live because something can still reach them. That ?something? may be obvious, or it may be hidden in a closure, cache, container, or traceback. When memory behavior looks strange, the right question is almost never ?why did Python randomly keep this alive?? The better question is ?what reference path still exists??


## Final Takeaway

The deepest way to revise **Object Lifetime, Memory, and Garbage Collection** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
